In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score

In [2]:
df = pd.read_csv('realistic_dummy_data.csv')
print(df.head())

   Hour  Minute  Day_of_Week  User_ID
0     1      55            0        2
1    20      24            5        3
2    16      23            2        3
3    22      49            3        5
4    20      18            5        1


In [3]:
# Add target column based on predefined rules
def is_usual_time(row):
    time_in_minutes = row["Hour"] * 60 + row["Minute"]
    day = row["Day_of_Week"]
    user = row["User_ID"]

    if user in [1, 2, 3, 4]:
        if day < 5:  # Weekday
            return 1 if 960 <= time_in_minutes <= 1200 else 0  # 16:00 to 20:00
        else:  # Weekend
            return 1 if time_in_minutes <= 60 else 0  # 00:00 to 01:00

    elif user == 5:  # User 5
        return 1 if 1320 <= time_in_minutes <= 1439 else 0  # 22:00 to 23:59

    return 0

In [4]:
df["Usual_Time"] = df.apply(is_usual_time, axis=1)

# Feature Engineering
df["time_in_minutes"] = df["Hour"] * 60 + df["Minute"]
X = df[["time_in_minutes", "Day_of_Week", "User_ID"]]
y = df["Usual_Time"]

In [5]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [6]:
# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        79
           1       1.00      1.00      1.00       121

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [7]:
# Perform 5-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5)

print("Cross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

Cross-Validation Scores: [0.995 0.995 1.    1.    1.   ]
Mean CV Accuracy: 0.998


In [8]:
# New data to test
new_data = {
    "Hour": [18, 23, 7, 16, 22],  # Add new test hours
    "Minute": [30, 50, 15, 45, 59],  # Add new test minutes
    "Day_of_Week": [0, 6, 2, 3, 5],  # Add new days (0=Monday, 6=Sunday)
    "User_ID": [1, 5, 3, 2, 4]  # Add new user IDs
}

new_df = pd.DataFrame(new_data)

# Preprocess new data
new_df["time_in_minutes"] = new_df["Hour"] * 60 + new_df["Minute"]

# Use the trained model to predict
new_df["Predicted_Usual_Time"] = model.predict(new_df[["time_in_minutes", "Day_of_Week", "User_ID"]])

# Compare against rule-based results (optional)
new_df["Rule_Based_Usual_Time"] = new_df.apply(is_usual_time, axis=1)

print(new_df)


   Hour  Minute  Day_of_Week  User_ID  time_in_minutes  Predicted_Usual_Time  \
0    18      30            0        1             1110                     1   
1    23      50            6        5             1430                     1   
2     7      15            2        3              435                     0   
3    16      45            3        2             1005                     1   
4    22      59            5        4             1379                     0   

   Rule_Based_Usual_Time  
0                      1  
1                      1  
2                      0  
3                      1  
4                      0  


In [11]:
# Check for mismatches between predictions and expected values
# Add expected values to the new data
new_df["Expected_Usual_Time"] = new_df.apply(is_usual_time, axis=1)

new_df["Prediction_Correct"] = new_df["Predicted_Usual_Time"] == new_df["Expected_Usual_Time"]

# Display mismatches, if any
mismatches = new_df[new_df["Prediction_Correct"] == False]
if mismatches.empty:
    print("All predictions are correct!")
else:
    print("Mismatched predictions:")
    print(mismatches)

All predictions are correct!


In [10]:
accuracy = (new_df["Prediction_Correct"].sum() / len(new_df)) * 100
print(f"Accuracy on new data: {accuracy:.2f}%")

Accuracy on new data: 100.00%


In [16]:
# Save predictions to a new CSV file
#df_test = X_test.copy()
#df_test['Predicted_Usual_Time'] = y_pred
#df_test.to_csv('predictions.csv', index=False)
#print("Predictions saved to predictions.csv")